# PTX Puzzles

Twelve small CUDA kernels, each missing one piece of synchronization. Fix each one between its `// >>> edit` and `// <<< edit` markers, then run the grade cell under it. [syncsan](https://github.com/ShreshthRajan/syncsan) grades every fix under the PTX memory model, so a fix that only happens to work on this GPU fails.

Pick a GPU runtime first (Runtime, then Change runtime type, then T4 GPU). Puzzle 8 needs an H100 or newer and is skipped on a T4.

In [ ]:
%%bash
# Setup, about a minute: install syncsan, build it for this GPU, fetch the puzzles.
pip -q install syncsan==0.10.5
syncsan build --arch $(nvidia-smi --query-gpu=compute_cap --format=csv,noheader | head -1 | tr -d .) > build.log 2>&1 && echo 'syncsan built' || tail -20 build.log
[ -d ptx-puzzles ] || git clone -q https://github.com/ShreshthRajan/ptx-puzzles
cd ptx-puzzles && python grade.py --help > /dev/null && echo 'puzzles ready'

## Puzzle 1: Message passing

In [ ]:
%%writefile ptx-puzzles/puzzles/01_message_passing.cu
// Puzzle 1: Message passing
//
// Block 1 writes 32 words and then sets a flag. Block 0 waits for the flag and then copies the words. Both flag
// accesses are relaxed. A relaxed flag carries its own value and orders nothing else, so block 0 can see the flag set
// and still read words from before block 1 wrote them.
//
// Make block 0's copy read block 1's words. Par: 2.
//
// Seen in: Hazy Research's Llama-1B megakernel, which hands activations between SMs through a relaxed counter and a
// volatile spin with no fence on either side (HazyResearch/Megakernels#16).
#include "common.cuh"

constexpr int N = 32;

__global__ void message_passing(unsigned* flag, unsigned* data, unsigned* out) {
  if (threadIdx.x != 0) return;
  if (blockIdx.x == 1) {                          // producer
    for (int i = 0; i < N; ++i) data[i] = i + 1;
    // >>> edit
    st_relaxed(flag, 1);
    // <<< edit
  } else {                                        // consumer
    // >>> edit
    while (ld_relaxed(flag) == 0) {}
    // <<< edit
    for (int i = 0; i < N; ++i) out[i] = data[i];
  }
}

int main() {
  unsigned* flag = device_zeros<unsigned>(1);
  unsigned* data = device_zeros<unsigned>(N);
  unsigned* out = device_zeros<unsigned>(N);
  message_passing<<<2, 32>>>(flag, data, out);
  finish_launch();
  unsigned got[N], want[N];
  host_copy(got, out, N);
  for (int i = 0; i < N; ++i) want[i] = i + 1;
  return check("out", got, want, N);
}


In [ ]:
!cd ptx-puzzles && python grade.py 1

## Puzzle 2: The volatile spin

In [ ]:
%%writefile ptx-puzzles/puzzles/02_volatile_spin.cu
// Puzzle 2: The volatile spin
//
// Block 1's threads each write one element of x. Thread 0 then fences and bumps a counter: that is a correct release.
// Block 0's thread 0 spins on the counter with a volatile load, the block syncs, and every thread reads x. A volatile
// load is a relaxed load. Nothing after the spin turns it into an acquire, so the reads of x are not ordered after
// block 1's writes.
//
// Make block 0's reads see block 1's writes. Par: 1.
//
// Seen in: megakernel-zoo's Qwen3-8B engine, Lucebox's Qwen3.5 decode kernel, and AutoMegaKernel (whose spin is
// atomicAdd(c, 0), commented as an acquire). All three released correctly and spun without a fence afterwards.
#include "common.cuh"

constexpr int T = 128;

__global__ void handoff(unsigned* ready, float* x, float* y) {
  const int t = threadIdx.x;
  if (blockIdx.x == 1) {                          // producer
    x[t] = 2.0f * t;
    __syncthreads();
    if (t == 0) { __threadfence(); atomicAdd(ready, 1u); }
  } else {                                        // consumer
    if (t == 0) {
      while (*(volatile unsigned*)ready == 0) {}
      // >>> edit
      // <<< edit
    }
    __syncthreads();
    y[t] = x[t] + 1.0f;
  }
}

int main() {
  unsigned* ready = device_zeros<unsigned>(1);
  float* x = device_zeros<float>(T);
  float* y = device_zeros<float>(T);
  handoff<<<2, T>>>(ready, x, y);
  finish_launch();
  float got[T], want[T];
  host_copy(got, y, T);
  for (int t = 0; t < T; ++t) want[t] = 2.0f * t + 1.0f;
  return check("y", got, want, T);
}


In [ ]:
!cd ptx-puzzles && python grade.py 2

## Puzzle 3: The missing release

In [ ]:
%%writefile ptx-puzzles/puzzles/03_missing_release.cu
// Puzzle 3: The missing release
//
// A triangular solve step. Four producer blocks each add a contribution into lsum with atomicAdd, then decrement a
// dependency counter, fmod. The consumer block spins until fmod reaches 0, fences, and reads lsum. The consumer side
// is a correct acquire. The producers decrement with a relaxed atomicSub and nothing before it, so their additions are
// not released: the consumer can see fmod at 0 and still read a partial sum.
//
// Make the consumer read the full sum. Par: 1.
//
// Seen in: SuperLU_DIST's GPU triangular solve. An earlier commit put back the consumer's fence after a port dropped
// it; the producer's release was still missing (xiaoyeli/superlu_dist#261, fixed upstream in 239280c).
#include "common.cuh"

constexpr int T = 128, P = 4;

__global__ void solve_step(int* fmod, float* lsum, float* x) {
  const int t = threadIdx.x;
  if (blockIdx.x > 0) {                           // producers 1..P
    atomicAdd(&lsum[t], (float)(blockIdx.x * t));
    __syncthreads();
    if (t == 0) {
      // >>> edit
      atomicSub(fmod, 1);
      // <<< edit
    }
  } else {                                        // consumer
    if (t == 0) { while (atomicAdd(fmod, 0) != 0) {} __threadfence(); }
    __syncthreads();
    x[t] = lsum[t] + 1.0f;
  }
}

int main() {
  const int deps = P;
  int* fmod = device_copy(&deps, 1);
  float* lsum = device_zeros<float>(T);
  float* x = device_zeros<float>(T);
  solve_step<<<1 + P, T>>>(fmod, lsum, x);
  finish_launch();
  float got[T], want[T];
  host_copy(got, x, T);
  for (int t = 0; t < T; ++t) want[t] = (float)(t * P * (P + 1) / 2) + 1.0f;
  return check("x", got, want, T);
}


In [ ]:
!cd ptx-puzzles && python grade.py 3

## Puzzle 4: The last block

In [ ]:
%%writefile ptx-puzzles/puzzles/04_last_block.cu
// Puzzle 4: The last block
//
// Each of 64 blocks sums its slice and thread 0 writes the block's partial. Thread 0 then takes a ticket from a
// counter. The block that draws the last ticket fences and adds up all 64 partials. The reader's side is a correct
// acquire. The writers take their tickets with a relaxed atomicAdd and nothing before it, so the last block can draw
// the last ticket and still read partials that were not yet written.
//
// Make the last block read every partial. Par: 1.
//
// Seen in: CUDA's threadFenceReduction sample and every reduction built on it (PyTorch's reduce_kernel among them). The
// fence before the ticket is the one that goes missing.
#include "common.cuh"

constexpr int B = 64, T = 128;

__device__ unsigned block_sum(unsigned* s) {      // sums s[0..T) in place; every thread gets the total
  for (int k = T / 2; k > 0; k >>= 1) { if (threadIdx.x < k) s[threadIdx.x] += s[threadIdx.x + k]; __syncthreads(); }
  return s[0];
}

__global__ void sum_last_block(const unsigned* in, unsigned* partial, unsigned* tickets, unsigned* total) {
  __shared__ unsigned s[T];
  __shared__ bool last;
  const int t = threadIdx.x, b = blockIdx.x;
  s[t] = in[b * T + t];
  __syncthreads();
  const unsigned mine = block_sum(s);
  if (t == 0) {
    partial[b] = mine;
    // >>> edit
    last = atomicAdd(tickets, 1u) == gridDim.x - 1;
    // <<< edit
  }
  __syncthreads();
  if (!last) return;
  if (t == 0) fence_acq_rel();                    // the reader's acquire: thread 0's ticket, then a fence
  __syncthreads();
  s[t] = t < B ? partial[t] : 0;
  __syncthreads();
  const unsigned all = block_sum(s);
  if (t == 0) *total = all;
}

int main() {
  unsigned h[B * T], want = 0;
  for (int i = 0; i < B * T; ++i) { h[i] = i % 7 + 1; want += h[i]; }
  unsigned* in = device_copy(h, B * T);
  unsigned* partial = device_zeros<unsigned>(B);
  unsigned* tickets = device_zeros<unsigned>(1);
  unsigned* total = device_zeros<unsigned>(1);
  sum_last_block<<<B, T>>>(in, partial, tickets, total);
  finish_launch();
  unsigned got;
  host_copy(&got, total, 1);
  return check("total", &got, &want, 1);
}


In [ ]:
!cd ptx-puzzles && python grade.py 4

## Puzzle 5: One thread signals for the block

In [ ]:
%%writefile ptx-puzzles/puzzles/05_block_release.cu
// Puzzle 5: One thread signals for the block
//
// Block 1's 128 threads each write one element of x. Thread 0 then fences and bumps a counter. Block 0 acquires the
// counter correctly and reads x. Thread 0's fence and atomic form a release, and a release orders the releasing
// thread's own earlier writes. It orders another thread's writes only if that thread synchronized with the releasing
// thread first. Warps 1 to 3 never did, so their elements are not published.
//
// Make every element of x reach block 0. Par: 1.
//
// Seen in: the usual megakernel signal, where one elected thread bumps the counter for its whole block; SuperLU_DIST's
// fix (239280c) adds a warp synchronization before the fence for the same reason.
#include "common.cuh"

constexpr int T = 128;

__global__ void handoff(unsigned* ready, float* x, float* y) {
  const int t = threadIdx.x;
  if (blockIdx.x == 1) {                          // producer
    x[t] = 3.0f * t;
    // >>> edit
    // <<< edit
    if (t == 0) { __threadfence(); atomicAdd(ready, 1u); }
  } else {                                        // consumer
    if (t == 0) { while (*(volatile unsigned*)ready == 0) {} __threadfence(); }
    __syncthreads();
    y[t] = x[t] - 1.0f;
  }
}

int main() {
  unsigned* ready = device_zeros<unsigned>(1);
  float* x = device_zeros<float>(T);
  float* y = device_zeros<float>(T);
  handoff<<<2, T>>>(ready, x, y);
  finish_launch();
  float got[T], want[T];
  host_copy(got, y, T);
  for (int t = 0; t < T; ++t) want[t] = 3.0f * t - 1.0f;
  return check("y", got, want, T);
}


In [ ]:
!cd ptx-puzzles && python grade.py 5

## Puzzle 6: One thread waits for the block

In [ ]:
%%writefile ptx-puzzles/puzzles/06_block_acquire.cu
// Puzzle 6: One thread waits for the block
//
// Block 1 publishes x correctly: every thread writes, the block syncs, thread 0 releases. In block 0, thread 0 spins and
// fences, which is an acquire. Warps 1 to 3 have their own work first (about 2 ms of it) and then read x. By then block
// 1 has long finished, so the reads look safe and on hardware they almost always are. Thread 0's acquire orders thread
// 0's later accesses. Warps 1 to 3 never synchronized with thread 0, so nothing orders their reads after block 1's
// writes.
//
// Make warps 1 to 3 read x after block 1 wrote it. Par: 1.
//
// Seen in: the usual megakernel wait, where one elected thread spins on the counter for its whole block.
#include "common.cuh"

constexpr int T = 128;

__device__ void other_work() { for (int i = 0; i < 20; ++i) __nanosleep(100000); }

__global__ void handoff(unsigned* ready, float* x, float* y) {
  const int t = threadIdx.x;
  if (blockIdx.x == 1) {                          // producer
    x[t] = 5.0f * t;
    __syncthreads();
    if (t == 0) { __threadfence(); atomicAdd(ready, 1u); }
  } else {                                        // consumer
    if (t == 0) { while (*(volatile unsigned*)ready == 0) {} __threadfence(); }
    else if (t >= 32) other_work();
    // >>> edit
    // <<< edit
    y[t] = x[t] + 2.0f;
  }
}

int main() {
  unsigned* ready = device_zeros<unsigned>(1);
  float* x = device_zeros<float>(T);
  float* y = device_zeros<float>(T);
  handoff<<<2, T>>>(ready, x, y);
  finish_launch();
  float got[T], want[T];
  host_copy(got, y, T);
  for (int t = 0; t < T; ++t) want[t] = 5.0f * t + 2.0f;
  return check("y", got, want, T);
}


In [ ]:
!cd ptx-puzzles && python grade.py 6

## Puzzle 7: The grid barrier's reset

In [ ]:
%%writefile ptx-puzzles/puzzles/07_grid_barrier.cu
// Puzzle 7: The grid barrier's reset
//
// A software grid barrier. Each block's thread 0 reads the generation, fences, and takes a ticket from a counter. The
// last block to arrive resets the counter, fences, and bumps the generation. The others spin on the generation and
// fence. The kernel runs 1,024 stencil steps with a barrier after each, so the counter is reset and reused. The
// arrivals release and the waits acquire. The reset is a plain store, and the last arriver's ticket was a relaxed
// atomic, so the reset is not ordered after the other blocks' tickets. That is a write-write race on the counter. No
// load reads a stale value, so adversarial mode has nothing to change here; the race report is the evidence. syncsan
// compares each write with the last write to its word, which is usually the last arriver's own ticket, so it sees the
// race only when two tickets land at nearly the same moment: about once in 40 crossings on an RTX 5090. The 1,024
// crossings make the report certain in practice.
//
// Make the reset safe. Par: 0.
//
// Seen in: Lucebox's Qwen3.5 decode megakernel (AtomicGridSync::sync, `*counter = 0`). RESOLVE reported the same race.
#include "common.cuh"

constexpr int B = 8, T = 128, N = B * T, STEPS = 1024;

__device__ void grid_sync(unsigned* count, unsigned* gen) {
  __syncthreads();
  if (threadIdx.x == 0) {
    const unsigned g = *(volatile unsigned*)gen;
    __threadfence();
    if (atomicAdd(count, 1u) == gridDim.x - 1) {
      // >>> edit
      *count = 0;
      // <<< edit
      __threadfence();
      atomicAdd(gen, 1u);
    } else {
      while (*(volatile unsigned*)gen == g) {}
      __threadfence();
    }
  }
  __syncthreads();
}

__global__ void stencil(unsigned* a, unsigned* b, unsigned* count, unsigned* gen) {
  const int i = blockIdx.x * T + threadIdx.x;
  for (int s = 0; s < STEPS; ++s) {
    b[i] = a[(i + N - 1) % N] + 2 * a[i] + a[(i + 1) % N] + s;   // neighbours at a block's edge live in another block
    grid_sync(count, gen);
    unsigned* c = a; a = b; b = c;
  }
}

int main() {
  unsigned h[N], tmp[N];
  for (int i = 0; i < N; ++i) h[i] = i * 2654435761u;
  unsigned* a = device_copy(h, N);
  unsigned* b = device_zeros<unsigned>(N);
  unsigned* count = device_zeros<unsigned>(1);
  unsigned* gen = device_zeros<unsigned>(1);
  stencil<<<B, T>>>(a, b, count, gen);
  finish_launch();
  for (int s = 0; s < STEPS; ++s) {
    for (int i = 0; i < N; ++i) tmp[i] = h[(i + N - 1) % N] + 2 * h[i] + h[(i + 1) % N] + s;
    for (int i = 0; i < N; ++i) h[i] = tmp[i];
  }
  unsigned got[N];
  host_copy(got, STEPS % 2 ? b : a, N);
  return check("a", got, h, N);
}


In [ ]:
!cd ptx-puzzles && python grade.py 7

## Puzzle 8: The bulk store's proxy fence

In [ ]:
%%writefile ptx-puzzles/puzzles/08_tma_store_fence.cu
// Puzzle 8: The bulk store's proxy fence (sm_90 and newer)
//
// Each block's threads write a tile into shared memory, the block syncs, and thread 0 copies the tile to global memory
// with the bulk-copy engine (the TMA unit's 1-D copy), then waits for it. The copy reads shared memory through the
// async proxy. __syncthreads() orders the threads' writes for other threads, which use the generic proxy. The async
// proxy needs its own fence between the generic writes and the copy, or the copy can read the tile as it was before.
//
// Make the copy read the finished tile. Par: 1.
//
// Seen in: every TMA store epilogue. CUTLASS calls this fence tma_store_fence() and puts it before the barrier.
#include "common.cuh"

constexpr int B = 4, T = 128, TILE = 256;

__global__ void tile_store(float* g) {
  __shared__ alignas(128) float tile[TILE];
  const int t = threadIdx.x, b = blockIdx.x;
  for (int i = t; i < TILE; i += T) tile[i] = (float)(b * 1000 + i);
  // >>> edit
  // <<< edit
  __syncthreads();
  if (t == 0) bulk_store_and_wait(g + b * TILE, tile, TILE * sizeof(float));
}

int main() {
  cudaDeviceProp prop;
  CK(cudaGetDeviceProperties(&prop, 0));
  if (prop.major < 9) { printf("needs sm_90 or newer (this GPU is sm_%d%d)\n", prop.major, prop.minor); return 77; }
  float* g = device_zeros<float>(B * TILE);
  tile_store<<<B, T>>>(g);
  finish_launch();
  static float got[B * TILE], want[B * TILE];
  host_copy(got, g, B * TILE);
  for (int b = 0; b < B; ++b) for (int i = 0; i < TILE; ++i) want[b * TILE + i] = (float)(b * 1000 + i);
  return check("g", got, want, B * TILE);
}


In [ ]:
!cd ptx-puzzles && python grade.py 8

## Puzzle 9: The reused reduction buffer

In [ ]:
%%writefile ptx-puzzles/puzzles/09_smem_reuse.cu
// Puzzle 9: The reused reduction buffer
//
// One block finds the argmax of each of 16 rows. For each row, every warp reduces its 32 values, lane 0 of each warp
// writes its best value and index into shared memory, the block syncs, and warp 0 reduces the 8 candidates and writes
// the row's answer. Then the loop moves to the next row and the warps write their candidates into the same slots. The
// barrier orders this row's writes before warp 0's reads. Nothing orders warp 0's reads before the next row's writes,
// so a warp that runs ahead can overwrite a slot that warp 0 has not read yet.
//
// Make every row's answer come from that row. Par: 1.
//
// Seen in: Mirage MPK's block_reduce_max_idx (tasks/ampere/argmax.cuh, and the sm100 copy), called once per batch
// row in a loop (mirage-project/mirage#843).
#include "common.cuh"

constexpr int T = 256, W = T / 32, ROWS = 16;

__device__ void warp_argmax(float& v, int& i) {
  for (int o = 16; o > 0; o >>= 1) {
    const float v2 = __shfl_down_sync(0xffffffffu, v, o);
    const int i2 = __shfl_down_sync(0xffffffffu, i, o);
    if (v2 > v) { v = v2; i = i2; }
  }
}

__global__ void argmax_rows(const float* x, int* out) {
  __shared__ float best_val[W];
  __shared__ int best_idx[W];
  const int t = threadIdx.x, lane = t % 32, warp = t / 32;
  for (int r = 0; r < ROWS; ++r) {
    float v = x[r * T + t];
    int i = t;
    warp_argmax(v, i);
    if (lane == 0) { best_val[warp] = v; best_idx[warp] = i; }
    __syncthreads();
    if (warp == 0) {
      v = lane < W ? best_val[lane] : -1.0f;
      i = lane < W ? best_idx[lane] : -1;
      warp_argmax(v, i);
      if (lane == 0) out[r] = i;
    }
    // >>> edit
    // <<< edit
  }
}

int main() {
  static float h[ROWS * T];
  int want[ROWS];
  for (int r = 0; r < ROWS; ++r)
    for (int t = 0; t < T; ++t) {
      h[r * T + t] = (float)((t * 37 + r * 11) % T);   // each row is a permutation of 0..255
      if ((t * 37 + r * 11) % T == T - 1) want[r] = t;
    }
  float* x = device_copy(h, ROWS * T);
  int* out = device_zeros<int>(ROWS);
  argmax_rows<<<1, T>>>(x, out);
  finish_launch();
  int got[ROWS];
  host_copy(got, out, ROWS);
  return check("out", got, want, ROWS);
}


In [ ]:
!cd ptx-puzzles && python grade.py 9

## Puzzle 10: The page goes back too early

In [ ]:
%%writefile ptx-puzzles/puzzles/10_page_release.cu
// Puzzle 10: The page goes back too early
//
// A megakernel block runs instructions out of a shared-memory page, with one warp per role. For the first instruction,
// the consumer (warp 0) writes its output O into the page's first half and arrives on O_READY. The storer (warp 1)
// waits there, copies O to global memory, and frees the page by arriving on FREE. The loader (warp 2) waits on FREE
// and loads the second instruction's input into the page. After arriving on O_READY, the consumer also writes the
// first instruction's log-sum-exp L into the page's second half. The page is freed when its reader is done; its last
// writer was never asked. The consumer's late write of L can land on top of the second instruction's input.
//
// Make the loader wait for every write into the page. Par: 1.
//
// Seen in: Hazy Research's Llama-1B megakernel, where the attention storer finishes the QOL page without waiting on
// L_arrived and the next instruction TMA-loads into it (HazyResearch/Megakernels#17).
#include "common.cuh"

constexpr int H = 32;                    // a page holds O[0..H) and then L[0..H)
constexpr int O_READY = 1, FREE = 2, LOADED = 3;
// >>> edit
constexpr int FREE_THREADS = 64;         // threads that meet at FREE: the storer and the loader
// <<< edit

__global__ void megakernel_block(const float* in1, const float* in2, float* out_o, float* out_l, float* out2) {
  __shared__ float page[2 * H];
  const int lane = threadIdx.x % 32, warp = threadIdx.x / 32;
  if (warp == 0) {                       // consumer
    const float a = in1[lane];
    page[lane] = 2.0f * a;               // O
    bar_arrive(O_READY, 64);
    const float l = a + 1.0f;            // L, computed after O went out
    page[H + lane] = l;
    out_l[lane] = l;
    // >>> edit
    // <<< edit
    bar_sync(LOADED, 64);                // second instruction
    out2[lane] = page[lane] * page[H + lane];
  } else if (warp == 1) {                // storer
    bar_sync(O_READY, 64);
    out_o[lane] = page[lane];
    bar_arrive(FREE, FREE_THREADS);
  } else {                               // loader
    bar_sync(FREE, FREE_THREADS);
    page[lane] = in2[lane];
    page[H + lane] = in2[H + lane];
    bar_arrive(LOADED, 64);
  }
}

int main() {
  float h1[H], h2[2 * H];
  for (int i = 0; i < H; ++i) h1[i] = (float)i;
  for (int i = 0; i < 2 * H; ++i) h2[i] = (float)(100 + i);
  float* in1 = device_copy(h1, H);
  float* in2 = device_copy(h2, 2 * H);
  float* out_o = device_zeros<float>(H);
  float* out_l = device_zeros<float>(H);
  float* out2 = device_zeros<float>(H);
  megakernel_block<<<1, 96>>>(in1, in2, out_o, out_l, out2);
  finish_launch();
  float got[H], want[H];
  int status = 0;
  host_copy(got, out_o, H); for (int i = 0; i < H; ++i) want[i] = 2.0f * h1[i];
  status |= check("out_o", got, want, H);
  host_copy(got, out2, H); for (int i = 0; i < H; ++i) want[i] = h2[i] * h2[H + i];
  status |= check("out2", got, want, H);
  return status;
}


In [ ]:
!cd ptx-puzzles && python grade.py 10

## Puzzle 11: Split-K, three ways

In [ ]:
%%writefile ptx-puzzles/puzzles/11_split_k.cu
// Puzzle 11: Split-K, three ways
//
// Three blocks each compute one split of the K loop of the same output tile. The first split stores its partial in a
// workspace and releases a flag. The middle split waits for the flag, adds its partial into the workspace with
// atomicAdd, and moves on. The final split waits for the same flag, reads the workspace, adds its own partial, and
// writes the tile. Each wait is a correct acquire. The final split acquires what the first split released and nothing
// more, so the middle split's additions are not ordered before its reads.
//
// Make the final split read all three partials. Par: 2.
//
// Seen in: CUTLASS's SM90 Stream-K GEMM before 3.7.0, where the final split waited for one earlier split with
// wait_lt(..., 1) and read the workspace while the middle split's atomicAdd was still landing (NVIDIA/cutlass#2027).
#include "common.cuh"

constexpr int T = 256, SPLITS = 3, K = 12, KS = K / SPLITS;

__global__ void splitk_tile(const float* a, unsigned* flag, float* work, float* out) {
  const int s = blockIdx.x, t = threadIdx.x;
  float p = 0.0f;
  for (int k = s * KS; k < (s + 1) * KS; ++k) p += a[k * T + t];
  if (s == 0) {                                   // first split
    work[t] = p;
    __syncthreads();
    if (t == 0) { __threadfence(); atomicAdd(flag, 1u); }
  } else if (s == 1) {                            // middle split
    if (t == 0) { while (*(volatile unsigned*)flag < 1) {} __threadfence(); }
    __syncthreads();
    for (int i = 0; i < 10; ++i) __nanosleep(100000);   // its K range is longer: Stream-K splits are uneven
    atomicAdd(&work[t], p);
    // >>> edit
    // <<< edit
  } else {                                        // final split
    if (t == 0) {
      // >>> edit
      while (*(volatile unsigned*)flag < 1) {}
      // <<< edit
      __threadfence();
    }
    __syncthreads();
    out[t] = work[t] + p;
  }
}

int main() {
  static float h[K * T];
  float want[T];
  for (int t = 0; t < T; ++t) {
    want[t] = 0.0f;
    for (int k = 0; k < K; ++k) { h[k * T + t] = (float)((k * 7 + t) % 13); want[t] += h[k * T + t]; }
  }
  float* a = device_copy(h, K * T);
  unsigned* flag = device_zeros<unsigned>(1);
  float* work = device_zeros<float>(T);
  float* out = device_zeros<float>(T);
  splitk_tile<<<SPLITS, T>>>(a, flag, work, out);
  finish_launch();
  float got[T];
  host_copy(got, out, T);
  return check("out", got, want, T);
}


In [ ]:
!cd ptx-puzzles && python grade.py 11

## Puzzle 12: The spinlock

In [ ]:
%%writefile ptx-puzzles/puzzles/12_spinlock.cu
// Puzzle 12: The spinlock
//
// 64 blocks each fold one value into a shared record (a sum, a count and a maximum) under a lock. Thread 0 takes the
// lock with atomicCAS, updates the record with plain loads and stores, and frees the lock with atomicExch. The lock
// gives mutual exclusion on the lock word only. Both atomics are relaxed, so the record's accesses inside one block's
// critical section are not ordered with those in the next. A block can read the record as it was before the previous
// holder's update and lose that update.
//
// Make the lock protect the record. Par: 2.
//
// Seen in: the textbook GPU spinlock, and every kernel that copies it with atomicCAS and atomicExch alone.
#include "common.cuh"

constexpr int B = 64;

struct Record { unsigned sum, count, max; };

__global__ void locked_update(unsigned* lock, Record* rec) {
  if (threadIdx.x != 0) return;
  const unsigned v = blockIdx.x * 37 % B + 1;
  while (atomicCAS(lock, 0u, 1u) != 0u) {}
  // >>> edit
  // <<< edit
  rec->sum += v;
  rec->count += 1;
  if (v > rec->max) rec->max = v;
  // >>> edit
  // <<< edit
  atomicExch(lock, 0u);
}

int main() {
  unsigned* lock = device_zeros<unsigned>(1);
  Record* rec = device_zeros<Record>(1);
  locked_update<<<B, 32>>>(lock, rec);
  finish_launch();
  Record got;
  host_copy(&got, rec, 1);
  unsigned g[3] = {got.sum, got.count, got.max}, want[3] = {0, B, 0};
  for (int b = 0; b < B; ++b) { const unsigned v = b * 37 % B + 1; want[0] += v; if (v > want[2]) want[2] = v; }
  return check("record", g, want, 3);
}


In [ ]:
!cd ptx-puzzles && python grade.py 12